In [2]:
import pandas as pd
import numpy as np
import sklearn
import pickle

In [3]:
print(f'pandas=={pd.__version__}')
print(f'numpy=={np.__version__}')
print(f'sklearn=={sklearn.__version__}')

pandas==3.0.3
numpy==2.4.6
sklearn==1.9.0


In [4]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

In [5]:
data_url = 'https://raw.githubusercontent.com/alexeygrigorev/mlbookcamp-code/master/chapter-03-churn-prediction/WA_Fn-UseC_-Telco-Customer-Churn.csv'

df = pd.read_csv(data_url)

df.columns = df.columns.str.lower().str.replace(' ', '_')

categorical_columns = list(df.dtypes[df.dtypes == 'str'].index)

for c in categorical_columns:
    df[c] = df[c].str.lower().str.replace(' ', '_')

df.totalcharges = pd.to_numeric(df.totalcharges, errors='coerce')
df.totalcharges = df.totalcharges.fillna(0)

df.churn = (df.churn == 'yes').astype(int)

In [6]:
y_train = df.churn

In [7]:
numerical = ['tenure', 'monthlycharges', 'totalcharges']

categorical = [
    'gender',
    'seniorcitizen',
    'partner',
    'dependents',
    'phoneservice',
    'multiplelines',
    'internetservice',
    'onlinesecurity',
    'onlinebackup',
    'deviceprotection',
    'techsupport',
    'streamingtv',
    'streamingmovies',
    'contract',
    'paperlessbilling',
    'paymentmethod',
]

In [9]:
for c in categorical:
    print(df[c].value_counts())
    print()

print('------------------')
for n in numerical:
    print(df[n].describe())
    print()

gender
male      3555
female    3488
Name: count, dtype: int64

seniorcitizen
0    5901
1    1142
Name: count, dtype: int64

partner
no     3641
yes    3402
Name: count, dtype: int64

dependents
no     4933
yes    2110
Name: count, dtype: int64

phoneservice
yes    6361
no      682
Name: count, dtype: int64

multiplelines
no                  3390
yes                 2971
no_phone_service     682
Name: count, dtype: int64

internetservice
fiber_optic    3096
dsl            2421
no             1526
Name: count, dtype: int64

onlinesecurity
no                     3498
yes                    2019
no_internet_service    1526
Name: count, dtype: int64

onlinebackup
no                     3088
yes                    2429
no_internet_service    1526
Name: count, dtype: int64

deviceprotection
no                     3095
yes                    2422
no_internet_service    1526
Name: count, dtype: int64

techsupport
no                     3473
yes                    2044
no_internet_service    15

In [7]:
from sklearn.pipeline import make_pipeline

In [8]:
pipeline = make_pipeline(
    DictVectorizer(sparse=False),
    LogisticRegression(solver='liblinear')
)

In [9]:
dv = DictVectorizer(sparse=False)

train_dict = df[categorical + numerical].to_dict(orient='records')

pipeline.fit(train_dict, y_train)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('dictvectorizer', ...), ('logisticregression', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
,"sparse sparse: bool, default=TrueWhether transform should produce scipy.sparse matrices.",False
,"dtype dtype: dtype, default=np.float64The type of feature values. Passed to Numpy array/scipy.sparse matrixconstructors as the dtype argument.",<class 'numpy.float64'>
,"separator separator: str, default=""=""Separator string used when constructing new features for one-hotcoding.",'='
,"sort sort: bool, default=TrueWhether ``feature_names_`` and ``vocabulary_`` should besorted when fitting.",True
Name,Type,Value


In [10]:
with open('model.bin', 'wb') as f_out:
    pickle.dump(pipeline, f_out)

In [11]:
with open('model.bin', 'rb') as f_in:
    pipeline = pickle.load(f_in)

In [13]:
customer = {
    'gender': 'male',
     'seniorcitizen': 0,
     'partner': 'no',
     'dependents': 'yes',
     'phoneservice': 'no',
     'multiplelines': 'no_phone_service',
     'internetservice': 'dsl',
     'onlinesecurity': 'no',
     'onlinebackup': 'yes',
     'deviceprotection': 'no',
     'techsupport': 'no',
     'streamingtv': 'no',
     'streamingmovies': 'no',
     'contract': 'month-to-month',
     'paperlessbilling': 'yes',
     'paymentmethod': 'electronic_check',
     'tenure': 6,
     'monthlycharges': 29.85,
     'totalcharges': 129.85
}
churn = pipeline.predict_proba(customer)[0, 1]

print(churn)
if churn >= 0.5:
    print('SEND AN EMAIL WITH PROMOTION')
else:
    print('EVERYTHING IS FINE')

0.5415461003027706
SEND AN EMAIL WITH PROMOTION
